# Unsupervised Learning — AI Engineering Interview Prep

Covers: K-Means, PCA, t-SNE, DBSCAN, hierarchical clustering.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits, make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, adjusted_rand_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

## 1. K-Means Clustering

In [ ]:
iris = load_iris()
X = StandardScaler().fit_transform(iris.data)
y_true = iris.target

# Elbow method to find optimal k
inertias = []
silhouettes = []
ks = range(2, 11)

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ks, inertias, 'bo-')
ax1.set_xlabel('k'); ax1.set_ylabel('Inertia'); ax1.set_title('Elbow Method')
ax2.plot(ks, silhouettes, 'go-')
ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette'); ax2.set_title('Silhouette Score')
plt.tight_layout()
plt.show()

# Fit with k=3
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = km3.fit_predict(X)
ari = adjusted_rand_score(y_true, labels)
print(f"k=3: silhouette={silhouette_score(X, labels):.3f}, ARI={ari:.3f}")

## 2. PCA for Dimensionality Reduction

In [ ]:
digits = load_digits()
X_d = digits.data  # 1797 x 64
X_d_s = StandardScaler().fit_transform(X_d)

pca = PCA()
pca.fit(X_d_s)

# Cumulative explained variance
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_components_95 = np.searchsorted(cumvar, 0.95) + 1

plt.figure(figsize=(10, 4))
plt.plot(cumvar, 'b-')
plt.axhline(0.95, color='r', linestyle='--', label='95% variance')
plt.axvline(n_components_95, color='g', linestyle='--', label=f'n={n_components_95}')
plt.xlabel('Number of Components'); plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — Explained Variance'); plt.legend()
plt.show()

print(f"Components to retain 95% variance: {n_components_95} (from {X_d.shape[1]})")

# Project to 2D and visualise
pca2 = PCA(n_components=2)
X_pca2 = pca2.fit_transform(X_d_s)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca2[:, 0], X_pca2[:, 1], c=digits.target,
                      cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter)
plt.title(f"PCA 2D — Digits (var={pca2.explained_variance_ratio_.sum():.1%})")
plt.show()

## 3. t-SNE Visualisation

In [ ]:
# t-SNE on digits (use PCA first to speed up)
X_pca50 = PCA(n_components=50).fit_transform(X_d_s)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=500).fit_transform(X_pca50)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=digits.target,
                      cmap='tab10', alpha=0.7, s=10)
plt.colorbar(scatter)
plt.title("t-SNE 2D — Digits (perplexity=30)")
plt.show()
print("t-SNE creates much cleaner clusters than PCA for digits!")

## 4. DBSCAN — Density-Based Clustering

In [ ]:
# Moons dataset — K-Means fails, DBSCAN succeeds
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

km_moons = KMeans(n_clusters=2, random_state=42)
db_moons = DBSCAN(eps=0.2, min_samples=5)

km_labels = km_moons.fit_predict(X_moons)
db_labels = db_moons.fit_predict(X_moons)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.scatter(X_moons[:,0], X_moons[:,1], c=km_labels, cmap='bwr', s=20)
ax1.set_title(f"K-Means (ARI={adjusted_rand_score(y_moons, km_labels):.2f})")

ax2.scatter(X_moons[:,0], X_moons[:,1], c=db_labels, cmap='bwr', s=20)
ax2.set_title(f"DBSCAN (ARI={adjusted_rand_score(y_moons, db_labels):.2f}, noise=-1)")

plt.tight_layout()
plt.show()
print(f"DBSCAN noise points: {(db_labels == -1).sum()}")

## Interview Tips

- **K-Means assumptions**: clusters are spherical, similar size, similar density. Fails on moons/rings.
- **K-Means initialization**: use `k-means++` (default in sklearn) to avoid bad random starts.
- **Silhouette score**: ranges [-1,1]; >0.5 = good separation; <0 = wrong cluster assignment.
- **PCA before t-SNE**: PCA to ~50 components first dramatically speeds up t-SNE.
- **t-SNE caveat**: distances between clusters are NOT meaningful — only local structure is preserved.
- **DBSCAN**: doesn't need k; handles noise (label=-1); struggles with varying density clusters.